# 05. LLM-As-A-Judge

이 노트북은 S0/S1/S2 출력에 대해 source-blind LLM-as-a-Judge 평가를 수행합니다.

Judge는 candidate가 pure/pair/metadata 중 어디서 왔는지 알지 못합니다. 입력은 case, gold, candidate, metadata이며, 출력은 Accept/Reject, confidence, social_context_needed, softened, scoring rubric입니다.

기본값은 `LIMIT = 20` smoke run입니다.

출력:

- `outputs/context_rag/judge_pure.csv`
- `outputs/context_rag/judge_pair.csv`
- `outputs/context_rag/judge_metadata.csv`


## judge 실행 설정

OpenAI model과 smoke run limit을 설정합니다. 이 노트북은 `judge_candidates()`를 직접 호출하며, 같은 작업은 `scripts/05_judge_candidates.py`로 command 실행할 수 있습니다.


In [ ]:
import pathlib, runpy

BOOTSTRAP = pathlib.Path("scripts/01_02_03_04_05_06_notebook_bootstrap.py")
if not BOOTSTRAP.exists():
    BOOTSTRAP = pathlib.Path("/content/lexnorm_submit/scripts/01_02_03_04_05_06_notebook_bootstrap.py")

setup_project = runpy.run_path(str(BOOTSTRAP))["setup_project"]
PROJECT_ROOT = setup_project()

from lexnorm.utils import sync_to_drive

from lexnorm.rag import judge_candidates

MODEL = "gpt-4.1-mini"
LIMIT = 20
CASES_CSV = "outputs/context_rag/audit_cases.csv"
METADATA_JSONL = "outputs/context_rag/metadata_cards.jsonl"
JUDGE_PROMPT = pathlib.Path("prompts/05_judge_system.txt")
JUDGE_SCHEMA = pathlib.Path("schemas/05_judge_schema.json")
JUDGE_TASKS = [
    ("outputs/context_rag/preds_pure.csv", "outputs/context_rag/judge_pure.csv"),
    ("outputs/context_rag/preds_pair.csv", "outputs/context_rag/judge_pair.csv"),
    ("outputs/context_rag/preds_metadata.csv", "outputs/context_rag/judge_metadata.csv"),
]


## 세 normalizer 출력에 대한 judge 평가

Pure, pair few-shot, metadata-RAG prediction을 각각 source-blind 방식으로 평가합니다. Judge prompt에는 candidate source를 넣지 않습니다.


In [ ]:
judge_dfs = {}
for preds_csv, output_csv in JUDGE_TASKS:
    print("=" * 80)
    print("judging", preds_csv, "->", output_csv)
    df = judge_candidates(
        cases_csv=CASES_CSV,
        preds_csv=preds_csv,
        output_csv=output_csv,
        prompt_path=JUDGE_PROMPT,
        schema_path=JUDGE_SCHEMA,
        model=MODEL,
        metadata_jsonl=METADATA_JSONL,
        limit=LIMIT,
    )
    judge_dfs[output_csv] = df
    print("saved", output_csv, len(df))
    display(df.head())
    sync_to_drive(output_csv)
